# Vežbe 7: Early stopping i uvod u konvolucione neuronske mreže

U ovom notebook-u nastavljamo rad sa **feedforward neuronskim mrežama** nad skupom slika **Fashion-MNIST** i uvodimo osnovne koncepte **konvolucionih neuronskih mreža**.

Prvi deo notebook-a predstavlja proširenje prethodnih vežbi, kroz uvođenje tehnike **early stopping** za kontrolu procesa treniranja modela.

Drugi deo notebook-a uvodi osnovne gradivne elemente konvolucionih neuronskih mreža, sa posebnim fokusom na:

- konvolucione slojeve,
- pooling slojeve,
- način na koji se kod slika čuva prostorna struktura podataka,
- razliku između feedforward i konvolucionih neuronskih mreža.


## 1. Import potrebnih biblioteka

In [ ]:
# Import potrebnih biblioteka

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


## 2. Postavljanje seed-a za reproduktivnost rezultata

In [ ]:
# Postaviti seed za reproduktivnost rezultata

torch.manual_seed(42)
split_generator = torch.Generator().manual_seed(42)

## 3. Učitavanje skupa podataka

In [ ]:
# Učitati trening i test skup Fashion-MNIST podataka

train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

# Prikazati broj slika u trening i test skupu

print("Broj slika u trening skupu:", len(train_dataset))
print("Broj slika u test skupu:", len(test_dataset))


## 4. Podela trening skupa na trening i validacioni deo

In [ ]:
# Podeliti trening skup na trening i validacioni deo

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_data, val_data = random_split(
    train_dataset,
    [train_size, val_size],
    generator=split_generator
)

print("Broj primera u trening skupu:", len(train_data))
print("Broj primera u validacionom skupu:", len(val_data))

## 5. Kreiranje batch-eva pomoću DataLoader-a

In [ ]:
# Formirati DataLoader objekte za trening, validacioni i test skup
# Koristiti batch_size = 32

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Prikazati broj batch-eva u trening, validacionom i test skupu

print("Broj batch-eva u trening skupu:", len(train_loader))
print("Broj batch-eva u validacionom skupu:", len(val_loader))
print("Broj batch-eva u test skupu:", len(test_loader))


## 6. Definisanje modela

In [ ]:
# Definisati model sa skrivenim slojevima i Dropout regularizacijom

class FeedForwardNetDropout(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)

        self.dropout = nn.Dropout(0.2)

    def forward(self, x):

        x = x.view(x.shape[0], -1)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))

        x = self.fc4(x)

        return x

## 7. Kreiranje instance modela

In [ ]:
# Kreirati instancu prethodno definisanog modela

model = FeedForwardNetDropout()

## 8. Definisanje funkcije greške i Adam optimizatora

In [ ]:
# Definisati funkciju greške CrossEntropyLoss

criterion = nn.CrossEntropyLoss()

# Definisati Adam optimizacioni algoritam

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

## 9. Early stopping

In [ ]:
# Definisati klasu EarlyStopping koja prati validation loss
# i prekida trening ako nema poboljšanja tokom zadatog broja epoha (patience)


In [ ]:
# Kreirati instancu klase EarlyStopping sa zadatim parametrima (patience=5, delta=0.001)


## 10. Trening modela uz early stopping

In [ ]:
# Istrenirati model uz primenu early stopping-a


In [ ]:
# Prikazati tabelarno train i validation loss po epohama


# Uvod u konvolucione neuronske mreže: konvolucioni i pooling slojevi

In [ ]:
# Uzeti jednu sliku iz trening skupa podataka
image, label = train_dataset[0]

# Dodati batch dimenziju: (1, 28, 28) -> (1, 1, 28, 28)
X = image.unsqueeze(0)

print("Dimenzije originalne slike:", image.shape)
print("Dimenzije slike sa batch dimenzijom:", X.shape)

In [ ]:
# Definisati jedan konvolucioni sloj


In [ ]:
# Primeniti konvolucioni sloj nad slikom
with torch.no_grad():
  feature_maps = conv(X)

In [ ]:
# Vizualizovati feature mape nakon prolaska kroz konvolucioni sloj
fig, ax = plt.subplots(2, 4, figsize=(10, 5))

for i in range(8):
    row = i // 4
    col = i % 4

    ax[row][col].imshow(feature_maps[0][i], cmap="gray")
    ax[row][col].set_title(f"Filter {i}")
    ax[row][col].axis("off")

plt.show()

In [ ]:
# Definisati ReLU aktivacionu funkciju


In [ ]:
# Primeniti ReLU aktivacionu funkciju nad feature mapama
with torch.no_grad():
    feature_maps_relu = relu(feature_maps)

In [ ]:
# Vizualizovati feature mape nakon primene ReLU funkcije

fig, ax = plt.subplots(2, 4, figsize=(10, 5))

for i in range(8):
    row = i // 4
    col = i % 4

    ax[row][col].imshow(feature_maps_relu[0][i], cmap="gray")
    ax[row][col].set_title(f"ReLU {i}")
    ax[row][col].axis("off")

plt.show()

In [ ]:
# Definisati MaxPooling sloj


In [ ]:
# Primeniti pooling sloj nad feature mapama posle ReLU
with torch.no_grad():
    pooled_maps = pool(feature_maps_relu)

In [ ]:
# Vizualizovati feature mape posle primene pooling sloja

fig, ax = plt.subplots(2, 4, figsize=(10, 5))

for i in range(8):
    row = i // 4
    col = i % 4

    ax[row][col].imshow(pooled_maps[0][i], cmap="gray")
    ax[row][col].set_title(f"Pool {i}")
    ax[row][col].axis("off")

plt.show()

In [ ]:
# Vizualizovati feature mape pre i posle pooling sloja
fig, ax = plt.subplots(2, 8, figsize=(16, 5))

for i in range(8):
    ax[0][i].imshow(feature_maps_relu[0][i], cmap="gray")
    ax[0][i].set_title(f"Pre {i}")
    ax[0][i].axis("off")

    ax[1][i].imshow(pooled_maps[0][i], cmap="gray")
    ax[1][i].set_title(f"Posle {i}")
    ax[1][i].axis("off")

plt.show()